# Taxonomy ergonomics

Phase-0 prerequisites for flows: hashable maps, bulk construction, a polars
frame view, and `Groups` from `group_by`.

Bulk construction and stepwise `stratify` build the same map. Two
equal maps are interchangeable as dict keys even though they are not
the same object. The bar chart is that map: one compartment for every
state × age pair. `group_by` hands back those same rows, one group
per pair.


In [ ]:
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import Groups, Property, PropertyMap

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))

pm = PropertyMap.from_properties([state, age])
assert len(pm) == 9
assert pm == PropertyMap.from_property(state).stratify(age)



## Hashable, so a cache key works

Rebuilding the map allocates a new object (`is not`) with the same rows
(`==`). The hash matches, so the new map retrieves the value stored under
the first one. There is nothing to plot here — the claim is identity of
content, not a trajectory.


In [ ]:
again = PropertyMap.from_properties([state, age])
assert pm is not again
assert pm == again
assert hash(pm) == hash(again)
assert {pm: "cached"}[again] == "cached"


## The frame is the compartment list

`to_frame()` is a polars table with one row per compartment and one column
per property. The grouped bars should be flat at 1: every state meets every
age exactly once. A missing bar would mean `from_properties` had dropped a
combination.


In [ ]:
frame = pm.to_frame()
assert frame.columns == ["state", "age"]
assert frame.height == len(pm)
assert set(frame["state"].to_list()) == {"S", "I", "R"}
frame.head(3)

tall = frame.group_by(["state", "age"]).len().to_pandas()
value_col = [col for col in tall.columns if col not in ("state", "age")][0]
wide = tall.pivot(index="state", columns="age", values=value_col)
assert int(wide.to_numpy().sum()) == len(pm)
wide.plot.bar(
    title="One compartment for every state × age",
    labels={"index": "state", "value": "compartments"},
)



## `group_by` is the same grid, as index arrays

The key `(state["I"], age["0-4"])` selects the same row as
`state["I"] & age["0-4"]`. The nine groups cover the map, which is the
table plotted above.


In [ ]:
groups = pm.group_by(state, age)
assert isinstance(groups, Groups)
assert len(groups) == 9
key = (state["I"], age["0-4"])
assert groups[key].tolist() == pm.select(state["I"] & age["0-4"]).tolist()
assert sum(idx.size for idx in groups.values()) == len(pm)